# Big Data Analytics — PySpark walkthrough

**Course:** Big Data Analytics — Apache Spark Analytics and Big Data Recommendation Systems  
**Dataset:** `Data Set/ecommerce_dataset_+1m.csv` (1,000,123 rows × 60 cols)  

This notebook demonstrates the **Part A** workflow end-to-end using **PySpark**:
1. Spark session setup
2. Distributed CSV ingestion
3. Spark SQL cleaning and transformations
4. Spark SQL aggregations / EDA
5. Spark MLlib classification (returned-order prediction)
6. Spark MLlib regression (profit prediction)

**Part B** (recommendation system, ALS + content-based + hybrid) is implemented in `src/recommender.py` and demonstrated through the Streamlit page `dashboard/pages/12_Recommendations.py`.

All helper functions live in `src/spark_session.py`, `src/spark_etl.py`, and `src/spark_models.py` so that the orchestrator script `run_spark_pipeline.py` re-uses the same logic shown here.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.spark_session import get_spark
spark = get_spark('notebook-eda')
spark

## 1. Load raw CSV with Spark

In [ ]:
from src import spark_etl as setl
raw = setl.load_csv(spark)
print('rows:', raw.count())
print('cols:', len(raw.columns))
raw.printSchema()

## 2. Clean + feature engineer

In [ ]:
df = setl.clean(raw).cache()
df.select('order_id', 'order_date', 'order_year', 'order_month', 'is_returned', 'is_high_fraud').show(5, truncate=False)

## 3. EDA via Spark SQL

Each aggregation runs as a distributed Spark job. We `toPandas()` only the small result set.

In [ ]:
import plotly.express as px

country = setl.kpi_country(df).toPandas()
px.bar(country.head(15).sort_values('revenue'), x='revenue', y='country', orientation='h',
       title='Top 15 countries by revenue').show()

In [ ]:
category = setl.kpi_category(df).toPandas()
px.treemap(category, path=['category', 'sub_category'], values='revenue', color='avg_margin',
           color_continuous_scale='RdYlGn', title='Revenue treemap (color = avg margin %)').show()

In [ ]:
daily = setl.kpi_daily(df).toPandas().sort_values('date')
px.line(daily, x='date', y=['revenue', 'profit'], title='Daily revenue and profit').show()

## 4. Spark MLlib — Classification (is_returned)

Pipeline: `Imputer` → `StringIndexer` → `OneHotEncoder` → `VectorAssembler` → `GBTClassifier`.

In [ ]:
from src.spark_models import train_spark_classifier
clf = train_spark_classifier(df.sample(0.1, seed=42), target='is_returned')
clf.metrics

## 5. Spark MLlib — Regression (profit_usd)

In [ ]:
from src.spark_models import train_spark_regressor
reg = train_spark_regressor(df.sample(0.1, seed=42), target='profit_usd')
reg.metrics

## 6. Recommendation system (Part B)

Below we just call the trained artefacts produced by `run_spark_pipeline.py`.
ALS was trained on **1,000,123 user-item interactions** with rank=24, then user/item factors were
exported to parquet for fast Spark-free inference.

In [ ]:
import pandas as pd
from src import recommender as rec

interactions = pd.read_parquet(rec.INTERACTIONS_PATH)
catalog = pd.read_parquet(rec.PRODUCT_CATALOG_PATH)
print(f'{len(interactions):,} interactions  /  {len(catalog):,} products')

# Use the most-active customer from the dataset (the data is sparse: ~1 order per customer).
top_customer = interactions.groupby('customer_id')['purchases'].sum().idxmax()
print('Demo customer:', top_customer)

In [ ]:
rec.recommend_als(None, top_customer, top_n=10)

In [ ]:
rec.recommend_content_for_user(top_customer, interactions, top_n=10)

In [ ]:
rec.recommend_hybrid(None, top_customer, interactions, top_n=10, alpha=0.6)

## 7. Stop the Spark session

In [ ]:
spark.stop()